# DetaNet training (task-specific)

Train a single task head at a time by selecting `TASK` below.


In [1]:
import os
import sys
from pathlib import Path

import torch
from torch_geometric.loader import DataLoader

CODE_DIR = Path.cwd()
if (CODE_DIR / 'detanet_model').exists() and str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from detanet_model import DetaNet, l1loss, l2loss, R2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_PATH = os.getenv('QM9S_PT', '../data/qm9s.pt')
TRAIN_ALL = True
TASK = 'npacharge'  # used only when TRAIN_ALL is False
TASKS_TO_TRAIN = None  # set to a list like ['npacharge', 'dipole'] to filter
MAX_ATOMIC_NUMBER = 118
BATCH_SIZE = 64
LR = 5e-4
WEIGHT_DECAY = 0.0
EPOCHS = 1
LIMIT = None  # set to an int for quick tests
SAVE_DIR = 'trained_param/latest'



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.12/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/home/ubuntu/venvs/pyg312/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/ubuntu/venvs/pyg312/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/ubuntu/venvs/pyg312/lib/p

In [2]:
!nvidia-smi

Wed Jan 14 04:39:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   24C    P0             63W /  300W |    2831MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
dataset = torch.load(DATASET_PATH, weights_only=False)
if LIMIT:
    dataset = dataset[:LIMIT]

print('dataset size:', len(dataset))
list(dataset[0].keys())
available_keys = set(dataset[0].keys())

train_datasets = []
val_datasets = []
for i, item in enumerate(dataset):
    if i % 20 == 0:
        val_datasets.append(item)
    else:
        train_datasets.append(item)

trainloader = DataLoader(train_datasets, batch_size=BATCH_SIZE, shuffle=True)
valloader = DataLoader(val_datasets, batch_size=BATCH_SIZE, shuffle=False)


dataset size: 129817


In [4]:
BASE_KWARGS = dict(
    num_features=128,
    act='swish',
    maxl=3,
    num_block=3,
    radial_type='trainable_bessel',
    num_radial=32,
    attention_head=8,
    rc=5.0,
    dropout=0.0,
    use_cutoff=False,
    max_atomic_number=MAX_ATOMIC_NUMBER,
    atom_ref=None,
    scale=1.0,
    norm=False,
)

TASKS = {
    'npacharge': dict(target='npacharge', model_kwargs=dict(
        scalar_outsize=1, irreps_out=None, summation=False, out_type='scalar', grad_type=None
    )),
    'dipole': dict(target='dipole', model_kwargs=dict(
        scalar_outsize=1, irreps_out='1o', summation=True, out_type='dipole', grad_type=None
    )),
    'polar': dict(target='polar', model_kwargs=dict(
        scalar_outsize=2, irreps_out='2e', summation=True, out_type='2_tensor', grad_type=None
    )),
    'quadrupole': dict(target='quadrupole', model_kwargs=dict(
        scalar_outsize=2, irreps_out='2e', summation=True, out_type='2_tensor', grad_type=None
    )),
    'hyperpolar': dict(target='hyperpolar', model_kwargs=dict(
        scalar_outsize=2, irreps_out='1o+3o', summation=True, out_type='3_tensor', grad_type=None
    )),
    'octapole': dict(target='octapole', model_kwargs=dict(
        scalar_outsize=2, irreps_out='1o+3o', summation=True, out_type='3_tensor', grad_type=None
    )),
    'Hi': dict(target='Hi', model_kwargs=dict(
        scalar_outsize=1, irreps_out=None, summation=False, out_type='scalar', grad_type='Hi'
    )),
    'Hij': dict(target='Hij', model_kwargs=dict(
        scalar_outsize=1, irreps_out=None, summation=False, out_type='scalar', grad_type='Hij'
    )),
    'dedipole': dict(target='dedipole', model_kwargs=dict(
        scalar_outsize=1, irreps_out='1o', summation=False, out_type='dipole', grad_type='dipole'
    )),
    'depolar': dict(target='depolar', model_kwargs=dict(
        scalar_outsize=2, irreps_out='2e', summation=False, out_type='2_tensor', grad_type='polar'
    )),
    'borden_os': dict(target='borden_os', model_kwargs=dict(
        scalar_outsize=240, irreps_out=None, summation=True, out_type='scalar', grad_type=None
    )),
    'shield_iso_c': dict(target='shield_iso_c', model_kwargs=dict(
        scalar_outsize=1, irreps_out=None, summation=False, out_type='scalar', grad_type=None
    )),
    'shield_iso_h': dict(target='shield_iso_h', model_kwargs=dict(
        scalar_outsize=1, irreps_out=None, summation=False, out_type='scalar', grad_type=None
    )),
}

def build_model(task):
    config = TASKS[task]
    model = DetaNet(**BASE_KWARGS, **config['model_kwargs'], device=DEVICE)
    model.to(DEVICE)
    model.train()
    return model, config


In [5]:
class Trainer:
    def __init__(self, model, train_loader, val_loader=None, loss_function=l2loss,
                 device=torch.device('cuda'), optimizer='Adam_amsgrad', lr=5e-4, weight_decay=0.0):
        self.opt_type = optimizer
        self.device = device
        self.model = model
        self.train_data = train_loader
        self.val_data = val_loader
        self.opts = {
            'AdamW': torch.optim.AdamW(self.model.parameters(), lr=lr, amsgrad=False, weight_decay=weight_decay),
            'AdamW_amsgrad': torch.optim.AdamW(self.model.parameters(), lr=lr, amsgrad=True, weight_decay=weight_decay),
            'Adam': torch.optim.Adam(self.model.parameters(), lr=lr, amsgrad=False, weight_decay=weight_decay),
            'Adam_amsgrad': torch.optim.Adam(self.model.parameters(), lr=lr, amsgrad=True, weight_decay=weight_decay),
            'Adadelta': torch.optim.Adadelta(self.model.parameters(), lr=lr, weight_decay=weight_decay),
            'RMSprop': torch.optim.RMSprop(self.model.parameters(), lr=lr, weight_decay=weight_decay),
            'SGD': torch.optim.SGD(self.model.parameters(), lr=lr, weight_decay=weight_decay),
        }
        self.optimizer = self.opts[self.opt_type]
        self.loss_function = loss_function
        self.step = -1

    def train(self, num_train, targ, stop_loss=1e-8, val_per_train=50, print_per_epoch=10):
        self.model.train()
        len_train = len(self.train_data)
        for _ in range(num_train):
            val_datas = iter(self.val_data) if self.val_data is not None else None
            for _, batch in enumerate(self.train_data):
                self.step += 1
                if self.device.type == 'cuda':
                    torch.cuda.empty_cache()
                self.optimizer.zero_grad()
                out = self.model(pos=batch.pos.to(self.device), z=batch.z.to(self.device),
                                 batch=batch.batch.to(self.device))
                target = batch[targ].to(self.device)
                loss = self.loss_function(out.reshape(target.shape), target)
                loss.backward()
                self.optimizer.step()

                if (self.step % val_per_train == 0) and (self.val_data is not None):
                    val_batch = next(val_datas)
                    val_target = val_batch[targ].to(self.device).reshape(-1)
                    val_out = self.model(pos=val_batch.pos.to(self.device), z=val_batch.z.to(self.device),
                                         batch=val_batch.batch.to(self.device)).reshape(val_target.shape)
                    val_loss = self.loss_function(val_out, val_target).item()
                    val_mae = l1loss(val_out, val_target).item()
                    val_R2 = R2(val_out, val_target).item()
                    if self.step % print_per_epoch == 0:
                        print('Epoch[{}/{}],loss:{:.8f},val_loss:{:.8f},val_mae:{:.8f},val_R2:{:.8f}'
                              .format(self.step, num_train * len_train, loss.item(), val_loss, val_mae, val_R2))
                    assert (loss > stop_loss) or (val_loss > stop_loss), (
                        'Training and prediction Loss is less than cut-off Loss, so training stops'
                    )
                elif (self.step % print_per_epoch == 0) and (self.step % val_per_train != 0):
                    print('Epoch[{}/{}],loss:{:.8f}'.format(self.step, num_train * len_train, loss.item()))

    def save_param(self, path):
        torch.save(self.model.state_dict(), path)


In [6]:
def train_task(task):
    config = TASKS[task]
    target = config['target']
    if target not in available_keys:
        print(f'skip {task}: target {target!r} not in dataset keys')
        return

    model, _ = build_model(task)
    trainer = Trainer(
        model,
        train_loader=trainloader,
        val_loader=valloader,
        loss_function=l2loss,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        optimizer='AdamW',
        device=DEVICE,
    )
    trainer.train(num_train=EPOCHS, targ=target)

    os.makedirs(SAVE_DIR, exist_ok=True)
    save_path = os.path.join(SAVE_DIR, f'latest_{task}.pth')
    trainer.save_param(save_path)
    print('saved', save_path)

    del model, trainer
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

if TRAIN_ALL:
    tasks = TASKS_TO_TRAIN or list(TASKS.keys())
    for task in tasks:
        train_task(task)
else:
    if TASK not in TASKS:
        raise ValueError(f'Unknown TASK {TASK!r}. Available: {sorted(TASKS)}')
    train_task(TASK)


Epoch[0/1927],loss:0.38627926,val_loss:0.16033280,val_mae:0.30855888,val_R2:-0.34617901
Epoch[10/1927],loss:0.07021379
Epoch[20/1927],loss:0.02644387
Epoch[30/1927],loss:0.01734471
Epoch[40/1927],loss:0.01415517
Epoch[50/1927],loss:0.01089390,val_loss:0.01013370,val_mae:0.06655730,val_R2:0.91463315
Epoch[60/1927],loss:0.00937126
Epoch[70/1927],loss:0.00713287
Epoch[80/1927],loss:0.00438361
Epoch[90/1927],loss:0.00315902
Epoch[100/1927],loss:0.00210857,val_loss:0.00275470,val_mae:0.03217318,val_R2:0.97141284
Epoch[110/1927],loss:0.00210222
Epoch[120/1927],loss:0.00174806
Epoch[130/1927],loss:0.00158567
Epoch[140/1927],loss:0.00134220
Epoch[150/1927],loss:0.00130520,val_loss:0.00232257,val_mae:0.03174773,val_R2:0.98322093
Epoch[160/1927],loss:0.00120847
Epoch[170/1927],loss:0.00128162
Epoch[180/1927],loss:0.00089548
Epoch[190/1927],loss:0.00070001
Epoch[200/1927],loss:0.00057651,val_loss:0.00078611,val_mae:0.01854091,val_R2:0.99385619
Epoch[210/1927],loss:0.00069878
Epoch[220/1927],loss: